# 03 · Live play — the REAL test  (the game API)

**Who Wants to Be a PoliMillionaire?** — here the actual game we play, not our dev set.

The `live`-mode counterpart of notebook 01, this is:
- **01 = OFFLINE** ('our own test'): the hand-crafted dev set, gold known, accuracy computed locally.
- **03 = LIVE** ('the real test'): the game server feeds the questions and grades them; `correct`
  only **after** submitting we learn (from `AnswerResult`).

Same pipeline, same `EvalRecord`/JSONL log — only `config.mode='live'` and a logged-in `GameClient` differ.
The single switch is `run_session(pipeline, config, game_client=...)` (D-015).

> ⚠️ **This plays a REAL game.** A leaderboard attempt it consumes and the **30s/question timer it starts**.
> The leaderboard resets ~1 week before the deadline → save serious runs for then. Run the play cell
> deliberately you should — not by a stray Shift+Enter.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 1 · Setup — clone the repo, add `millionaire_client` to the path, install deps
The same clone as notebook 01, plus the course's `millionaire_client` package onto `sys.path` we add
(in the repo at `NLP_assignment_api_client/` it lives).

In [16]:
import os, sys

# The repo, into Colab we clone -- present already? Then update it instead, we do.
REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
if not os.path.exists(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull -q

# Our src tree AND the provided client package, onto the import path both go.
SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
    if p not in sys.path:
        sys.path.insert(0, p)

# Into the repo root, change directory we do -- relative paths simpler they become.
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('On sys.path:', SRC, '|', API_CLIENT)

# The provided client, importable it must be -- confirm we do, before any game touch.
from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

Repo root: /content/NLP
On sys.path: /content/NLP/src | /content/NLP/NLP_assignment_api_client
millionaire_client, imported it is.


In [17]:
# The inference stack + the client's `requests`, install we do (light it stays).
!pip install -q 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.43.0' sentencepiece einops pyyaml pandas matplotlib requests
print('Installed, the dependencies are.')

Installed, the dependencies are.


In [18]:
import torch

# A GPU, present it must be -- on CPU, a 7B model in 30s answer we cannot.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING -- no GPU. Runtime ▸ Change runtime type ▸ T4 GPU, switch you must.')

CUDA available: True
GPU: Tesla T4


## 2 · Load the LIVE run config
`configs/live.yaml` carries `mode: 'live'`. The competition + game mode, here you choose.

Competitions: 0 Entertainment · 1 History · 2 Science · 3 Maths · 4 Philosophy · 5 News.

In [19]:
from config import RunConfig

# The live config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# Which competition to play + how, here choose it you do.
config.game.competition_id = 0          # 0..5
config.game.game_mode = 'text'          # 'text' | 'speech'
config.run_id = f'live_comp{config.game.competition_id}'

print('mode:', config.mode)
print('competition_id:', config.game.competition_id, '| game_mode:', config.game.game_mode)
print('aim_seconds:', config.game.aim_seconds, '(below the 30s wall, a network margin this keeps)')
print('model:', config.model.name, '|', config.model.quantization)

mode: live
competition_id: 0 | game_mode: text
aim_seconds: 25.0 (below the 30s wall, a network margin this keeps)
model: Qwen/Qwen2.5-7B-Instruct | 4bit


## 3 · Load + warm up the model
Identical to notebook 01 — the cold-start cost paid **before** any timed question, so the 30s wall a cold load never eats.

In [25]:
import time
from inference.engine import TransformersEngine

# Once, the model we load -- the cold-start cost, here we pay it.
t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine 已在显存中,跳过加载。')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 85.81 MiB is free. Including non-PyTorch memory, this process has 14.48 GiB memory in use. Of the allocated memory 14.16 GiB is allocated by PyTorch, and 196.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 4 · Wire the pipeline (same as offline) + log in to the game
DI exactly as notebook 01 (D-006) — only now a logged-in `GameClient` we also build.

Credentials: a PoliMi email as the username; the password from a **Colab secret** named
`poli-millionaire` we read (hardcode it, we must NOT — B3). Add it via the 🔑 panel on the left.

In [ ]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline

# The collaborators, injected into the pipeline they are (D-006) -- identical to offline.
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
    classifier=QuestionClassifier(),
    retriever=None,   # Off for now -- Phase 4 turns it on.
    tools=None,       # Off for now -- Phase 3 turns it on.
    latency_budget_s=config.latency_budget_s,
)

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = 'runjie dai'        # <-- your PoliMi email, fill it you must.
PASSWORD = userdata.get('password')  # <-- a Colab secret, NOT hardcoded (B3).

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions and their ids, list them we do (safe -- no timer this starts).
for c in game_client.list_competitions():
    ml = getattr(c, 'max_levels', '?')
    print('  id=', c.id, '|', c.name, '| max_levels=', ml)

## 5 · ▶ Play ONE real game  (consumes a leaderboard attempt + starts the 30s timer)
`run_session` sees `mode='live'` and drives `LiveRunner`: start game → our pipeline answers each
question → submit by integer `Option.id` → log one `EvalRecord` per turn (with `correct` from the server).

⚠️ Deliberately run this. Each turn prints live; the full per-turn record lands in the JSONL log.

In [ ]:
from evaluation.runner import run_session

# The ONE switch -- mode='live' it sees, so LiveRunner over the real game it drives.
run_path = run_session(
    pipeline,
    config,
    game_client=game_client,
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
)
print('Live run written to:', run_path)

## 6 · Results — how far did we get?
From the same JSONL log as offline it derives. `correct` is None where the server withheld it
(e.g. a timeout). Highest level reached + a per-turn table, here we show.

In [ ]:
from evaluation.metrics import load_runs

# The live run, back into a DataFrame we read.
df = load_runs([run_path])

answered = len(df)
known = df[df['correct'].notna()]
n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
acc = (n_correct / len(known)) if len(known) else float('nan')
reached = int(df['level'].dropna().max()) if df['level'].notna().any() else None

print(f'Questions answered: {answered}')
print(f'Correct: {n_correct}/{len(known)}  (accuracy {acc:.0%})')
print(f'Highest level reached: {reached}')

# Per question, the full card -- the choices too, so the wrong ones inspect we can
# (the picked letter, by its option text now we read). Note: `options` only the live
# runs from NOW ON carry; an older log an empty cell shows.
for _, row in df.iterrows():
    opts = row['options'] if 'options' in df.columns and isinstance(row['options'], dict) else {}
    picked = row['predicted_answer']
    mark = {True: 'CORRECT', False: 'WRONG'}.get(row['correct'], 'UNKNOWN')  # None (server withheld) it is.
    print('\n' + '=' * 70)
    print(f"[{mark}] qid={row['qid']} | level={row['level']} | topic={row['topic']} | latency={row['latency_s']:.1f}s")
    print(f"Q: {row['question_text']}")
    if opts:
        for letter, text in opts.items():
            arrow = '   <-- model picked' if letter == picked else ''
            print(f"   {letter}. {text}{arrow}")
    else:
        print(f"   (options not logged for this run)  model picked: {picked}")
    print(f"correct={row['correct']}  |  confidence={row['confidence']}")


In [24]:
import gc, torch
for name in ('engine', 'pipeline'):
      if name in globals():
          del globals()[name]
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'free {free/1e9:.1f} GiB / total {total/1e9:.1f} GiB')

free 6.0 GiB / total 15.6 GiB


## 7 · Observations + next steps

_Fill in after the first real game (feeds the investigation rubric + confirms the API contract):_

- **API contract check (Phase-0 TODO):** did `Option.id` submission + `time_remaining` behave as documented?
- **Latency under the real wall:** any turn near 30s once network RTT is counted? Tune `game.aim_seconds`.
- **Where we fall:** the level + topic of the first wrong answer — a knowledge gap, or a parse slip?
- **Offline vs live gap:** does live accuracy track the ~87% the dev set predicted, or drift?

**Switching modes is a one-liner:** offline ⇄ live is just `config.mode` + whether you pass `game_client`.
For a quick offline re-check: `run_session(pipeline, RunConfig.from_yaml('configs/base.yaml'))` — no game_client.

> Be polite to the server (rate limits are real). Save real leaderboard pushes for after the ~1-week-pre-deadline reset.